# Simple Linear Regression Practical - Weight to Height

**Goal:** use a person's weight to predict height. This is simple linear regression because there is **one input** and **one numeric output**.

## 1. What this notebook does

1. Load the supplied height-weight data.
2. Look at the relationship with a scatter plot.
3. Split data into training and test sets.
4. Train a simple linear regression model.
5. Check its errors and make a new prediction.
6. Compare scikit-learn's result with the OLS formula.

The 23 rows from the attached CSV are kept inside this notebook, so the code runs without needing a separate local file.

## 2. Load and inspect the data

- **Weight** is the input feature ($X$).
- **Height** is the target we want to predict ($y$).

The code below creates a small pandas table from the supplied CSV values, then shows the first rows and checks for missing values.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

# Values copied from the supplied height-weight.csv file
data = {
    'Weight': [45, 58, 48, 60, 70, 78, 80, 90, 95, 78, 82, 95, 105, 100, 85, 78, 50, 65, 76, 87, 45, 56, 72],
    'Height': [120, 135, 123, 145, 160, 162, 163, 175, 182, 170, 176, 182, 175, 183, 170, 177, 140, 159, 150, 167, 129, 140, 160]
}
df = pd.DataFrame(data)

print('Rows, columns:', df.shape)
print('Missing values in each column:')
print(df.isna().sum())
df.head()


## 3. Is a straight line reasonable?

A scatter plot lets us see the data as dots. If the dots mostly rise from left to right, a positive straight-line relationship may be useful. Correlation close to +1 means a strong positive linear relationship; it does **not** prove that one variable causes the other.

In [ ]:
correlation = df['Weight'].corr(df['Height'])

plt.figure(figsize=(7, 4.5))
plt.scatter(df['Weight'], df['Height'], color='#1d3557', s=60)
plt.title(f'Weight and height: correlation = {correlation:.2f}', weight='bold')
plt.xlabel('Weight')
plt.ylabel('Height')
plt.grid(alpha=0.2)
plt.show()

print(f'Correlation: {correlation:.3f}')
print('The positive value means larger weights usually appear with larger heights in this small dataset.')


## 4. Split the data and train the model

- `X` uses double brackets, so it stays a 2D table with one column. scikit-learn expects features in this shape.
- `y` is a 1D target column.
- `test_size=0.25` keeps about one quarter of the rows for an honest final check.
- `random_state=42` makes this learning example repeatable.

We do **not** standardize weight here. Ordinary one-feature `LinearRegression` can learn directly from the original units, which also makes the slope easier to explain.

In [ ]:
# X is a 2D feature table; y is the 1D target column
X = df[['Weight']]
y = df['Height']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

model = LinearRegression()
model.fit(X_train, y_train)

slope = float(model.coef_[0])
intercept = float(model.intercept_)
print('Training feature shape:', X_train.shape)
print('Test feature shape:', X_test.shape)
print(f'Learned equation: predicted height = {intercept:.2f} + ({slope:.2f} x weight)')
print(f'Easy meaning: adding 1 weight unit changes predicted height by about {slope:.2f} units.')


## 5. Test the model

Now the model sees the held-out test weights and predicts their heights.

- **MAE:** average size of a mistake, in height units.
- **RMSE:** also in height units, but gives extra importance to large mistakes.
- **R-squared:** how much better the model is than always predicting the average height.
- **Adjusted R-squared:** R-squared with a penalty for unnecessary input features. Here there is only one input, so it should be close to R-squared.

With only 23 rows, test scores can move a lot when the split changes. Treat this as a practice dataset, not a real-world accuracy claim.

In [ ]:
y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)
n_test_rows = len(y_test)
number_of_features = X_test.shape[1]
adjusted_r2 = 1 - (1 - r2) * (n_test_rows - 1) / (n_test_rows - number_of_features - 1)

metrics = pd.Series({
    'MAE (height units)': mae,
    'MSE (height units squared)': mse,
    'RMSE (height units)': rmse,
    'R-squared': r2,
    'Adjusted R-squared': adjusted_r2
}).round(3)
metrics


## 6. Visual check: training data, test data, and residuals

The orange line is the model. Green vertical lines show test residuals: the gap between a real test height and the predicted height. Smaller gaps mean better predictions.

In [ ]:
# Draw one line across the full weight range
weight_line = pd.DataFrame({'Weight': np.linspace(df['Weight'].min(), df['Weight'].max(), 100)})
height_line = model.predict(weight_line)

plt.figure(figsize=(8, 5))
plt.scatter(X_train['Weight'], y_train, color='#1d3557', s=60, label='Training data', zorder=3)
plt.scatter(X_test['Weight'], y_test, color='#d62828', s=70, label='Test data', zorder=3)
plt.plot(weight_line['Weight'], height_line, color='#f77f00', linewidth=2.5, label='Learned line')
for weight, actual, predicted in zip(X_test['Weight'], y_test, y_pred):
    plt.vlines(weight, min(actual, predicted), max(actual, predicted), color='#2a9d8f', linewidth=2)
plt.title('Simple linear regression: weight to height', weight='bold')
plt.xlabel('Weight')
plt.ylabel('Height')
plt.legend()
plt.grid(alpha=0.2)
plt.show()


## 7. OLS check: the same idea without scikit-learn

Ordinary Least Squares (OLS) calculates the slope and intercept from formulas that minimize squared residuals. For simple linear regression:

`slope = sum((x - x_mean) * (y - y_mean)) / sum((x - x_mean)^2)`

`intercept = y_mean - slope * x_mean`

The next cell calculates these from the same training data and compares them with scikit-learn. They should match, apart from tiny rounding differences.

In [ ]:
x_train_values = X_train['Weight'].to_numpy()
y_train_values = y_train.to_numpy()
x_mean = x_train_values.mean()
y_mean = y_train_values.mean()

ols_slope = np.sum((x_train_values - x_mean) * (y_train_values - y_mean)) / np.sum((x_train_values - x_mean) ** 2)
ols_intercept = y_mean - ols_slope * x_mean

print(f'Manual OLS slope: {ols_slope:.6f}')
print(f'scikit-learn slope: {slope:.6f}')
print(f'Manual OLS intercept: {ols_intercept:.6f}')
print(f'scikit-learn intercept: {intercept:.6f}')
print('Do the methods match?', np.isclose(ols_slope, slope) and np.isclose(ols_intercept, intercept))


## 8. Predict one new value

For a new input, keep the same column name and 2D table shape used during training. Here we ask the model to predict height for weight 72.

In [ ]:
new_person = pd.DataFrame({'Weight': [72]})
predicted_height = model.predict(new_person)[0]
print(f'For weight 72, predicted height = {predicted_height:.2f}')


## 9. Quick revision

- Keep features in `X` as a 2D table: `df[['Weight']]`.
- Keep the prediction target in `y`: `df['Height']`.
- Train only on training data; evaluate only on unseen test data.
- The slope tells how the predicted height changes when weight increases by one unit.
- OLS and scikit-learn Linear Regression solve the same squared-error problem for this simple case.

**One-line answer:** This model learns a best-fit line from weight and uses it to estimate height for a new weight.